# Aplicação de *Rapidly-exploring Random Trees* (RRT) no Complexo Industrial NPK (`R-101`)
**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta (UNIFEI)  
**Unidade de Processo:** Setor 100 — Reator *Pre-Neutralizer* (`R-101`) e Sistema Instrumentado de Segurança (`SCADA-Core`)

---

### 1. Contextualização do Processo e Objetivo de Controle

No Complexo Industrial de Fertilizantes Químicos NPK, o reator **Pre-Neutralizer (`R-101`)** realiza a neutralização fortemente exotérmica entre amônia anidra líquida ($\text{NH}_3$) e ácido fosfórico ($\text{H}_3\text{PO}_4$) para produzir *slurry* de **MAP** (Fosfato Monoamônico) ou **DAP** (Fosfato Diamônico):

1. **Síntese de MAP:**
   $$\text{NH}_3\text{ (liq)} + \text{H}_3\text{PO}_4\text{ (aq)} \longrightarrow \text{NH}_4\text{H}_2\text{PO}_4\text{ (slurry)} \quad (\Delta H^\circ = -135\text{ kJ/mol})$$
2. **Síntese de DAP:**
   $$2\,\text{NH}_3\text{ (liq)} + \text{H}_3\text{PO}_4\text{ (aq)} \longrightarrow (\text{NH}_4)_2\text{HPO}_4\text{ (slurry)} \quad (\Delta H^\circ = -228\text{ kJ/mol})$$

**Problema de Controle:** Deseja-se realizar a transição de campanha produtiva de **MAP** (razão molar $\alpha = \text{N/P} \approx 1{,}02$, baixa carga térmica) para **DAP** ($\alpha \approx 1{,}85$, alta produtividade e maior liberação de calor) sem violar nenhum intertravamento do **Sistema Instrumentado de Segurança (SIS)** definido pelas tags ISA-5.1 (`PT-101`, `TT-101`, `AT-101`, `LT-101A/B`).

In [1]:
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Configuração visual dos gráficos
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# Catálogo Oficial de Setpoints Críticos do SIS (Norma ISA-5.1)
SIS_LIMITS = {
    "P_max": 180.0,      # PT-101: Sobrepressão Crítica (P >= 180 bar) -> p1 = 1
    "T_max": 200.0,      # TT-101: Sobretemperatura (T >= 200 °C) -> t1 = 1
    "C_nh3_max": 25.0,   # AT-101: Vazamento Tóxico (C_NH3 >= 25 ppm) -> g1 = 1
    "L_max": 90.0,       # LT-101A: Risco Transbordamento (Nível >= 90%) -> l_high = 1
    "L_min": 20.0,       # LT-101B: Risco Cavitação (Nível <= 20%) -> l_low = 1
}

print("Parâmetros do SIS (ISA-5.1) carregados com sucesso:")
for tag, val in SIS_LIMITS.items():
    print(f"  - {tag:10s}: {val}")

Parâmetros do SIS (ISA-5.1) carregados com sucesso:
  - P_max     : 180.0
  - T_max     : 200.0
  - C_nh3_max : 25.0
  - L_max     : 90.0
  - L_min     : 20.0


### 2. Modelagem Híbrida do Processo e Barreira Lógica ($\Phi_{\text{SIS}}$)

O estado do reator `R-101` pertence ao espaço híbrido $\mathcal{Z} = \mathcal{X} \times \mathcal{Q}$:
- **Vetor de Estados Contínuos $\mathbf{x} = [P, T, L, C_{\text{NH}_3}, \alpha]^T \in \mathbb{R}^5$:**
  - $P$: Pressão interna medida por `PT-101` ($\text{bar}$)
  - $T$: Temperatura do *slurry* medida por `TT-101` ($^\circ\text{C}$)
  - $L$: Nível volumétrico medido por `LT-101A/B` ($\%$)
  - $C_{\text{NH}_3}$: Concentração ambiental de amônia medida por `AT-101` ($\text{ppm}$)
  - $\alpha$: Razão molar estequiométrica $\text{N/P}$ no meio reacional
- **Vetor de Estados Discretos $\mathbf{q} = (v_1, v_2, m_1) \in \{0, 1\}^3$:**
  - $v_1$: Válvula solenoide de bloqueio de $\text{NH}_3$ (`XV-101`)
  - $v_2$: Válvula solenoide de bloqueio de $\text{H}_3\text{PO}_4$ (`XV-102`)
  - $m_1$: Motor do agitador de pás (`AG-101`) — quando desligado ($m_1=0$), a mistura cai para $25\%$, favorecendo acúmulo local de amônia não reagida.

A função booleana do intertravamento de segurança é dada pela disjunção lógica:
$$\Phi_{\text{SIS}}(\mathbf{z}) = p_1 \lor t_1 \lor g_1 \lor l_{\text{high}} \lor l_{\text{low}} \lor e_1$$
Qualquer trajetória que atinja $\Phi_{\text{SIS}}(\mathbf{z}) = 1$ pertence ao conjunto de obstáculos $\mathcal{Z}_{\text{obs}}$ e é imediatamente podada pelo RRT.

In [2]:
def avaliar_sis_booleano(x, e1=0):
    """
    Avalia as proposições lógicas ISA-5.1 e retorna (Phi_SIS, dict_proposicoes).
    Phi_SIS == 1 indica violação de segurança (estado em Z_obs).
    """
    P, T, L, C_nh3, alpha = x
    props = {
        "p1": int(P >= SIS_LIMITS["P_max"]),
        "t1": int(T >= SIS_LIMITS["T_max"]),
        "g1": int(C_nh3 >= SIS_LIMITS["C_nh3_max"]),
        "l_high": int(L >= SIS_LIMITS["L_max"]),
        "l_low": int(L <= SIS_LIMITS["L_min"]),
        "e1": int(e1 == 1),
    }
    phi_sis = int(any(props.values()))
    return phi_sis, props


def dinamica_reator_r101(x, q, u_cont):
    """
    Sistema de EDOs não lineares dx/dt = f(x, q, u) do Reator R-101.
    x      = [P (bar), T (°C), L (%), C_nh3 (ppm), alpha (N/P)]
    q      = (v1, v2, m1) -> estados discretos (XV-101, XV-102, AG-101)
    u_cont = [u_nh3, u_h3po4, u_cool] em [0, 1]^3
    """
    P, T, L, C_nh3, alpha = x
    v1, v2, m1 = q
    u1, u2, u3 = u_cont

    # Vazões efetivas condicionadas ao estado das válvulas de bloqueio
    F_N = v1 * u1
    F_P = v2 * u2
    F_out = u3

    # Eficiência de mistura (agitador AG-101 ligado vs. desligado)
    eta_mix = 1.0 if m1 == 1 else 0.25

    # Taxa de reação exotérmica de neutralização
    r_rxn = eta_mix * min(F_N, 2.0 * F_P) * (1.0 + 0.004 * max(0.0, T - 100.0))

    # Entalpia efetiva interpolada entre MAP (135 kJ/mol) e DAP (228 kJ/mol)
    dH_eff = 135.0 + 93.0 * np.clip(alpha - 1.0, 0.0, 1.0)

    # 1. Balanço de Energia -> dT/dt (°C/min)
    dT = 0.28 * dH_eff * r_rxn - 0.14 * (T - 45.0) - 18.0 * F_out

    # 2. Pressão de Vapor + Acúmulo de Amônia -> dP/dt (bar/min)
    P_eq = 45.0 + 0.0032 * (T ** 2)
    dP = 0.22 * (P_eq - P) + 18.0 * F_N * (1.15 - 0.5 * eta_mix) - 14.0 * F_out

    # 3. Balanço Volumétrico -> dL/dt (%/min)
    dL = 6.5 * (F_N + 1.2 * F_P) - 9.5 * F_out * (L / 50.0)

    # 4. Emissão Fugitiva de NH3 -> dC_nh3/dt (ppm/min)
    excesso_N = max(0.0, F_N - 1.6 * F_P)
    sobre_P = max(0.0, P - 135.0)
    dC = 16.0 * excesso_N + 0.08 * sobre_P + (1 - m1) * 12.0 * F_N - 0.35 * C_nh3

    # 5. Dinâmica da Razão Molar N/P -> d_alpha/dt (1/min)
    alpha_feed = (F_N + 0.05) / (F_P + 0.05)
    d_alpha = 0.18 * (F_N + F_P + 0.2) * (np.clip(alpha_feed, 0.5, 2.3) - alpha)

    return np.array([dP, dT, dL, dC, d_alpha], dtype=float)


def integrar_passo_seguro(x0, q, u_cont, dt=1.0, n_sub=10, ignorar_sis=False):
    """
    Integra a trajetória via Runge-Kutta de 4ª Ordem (RK4) e verifica
    continuamente se Phi_SIS == 0 em todos os pontos intermediários.
    """
    h = dt / n_sub
    x = x0.copy()
    traj = [x.copy()]

    if not ignorar_sis:
        phi, _ = avaliar_sis_booleano(x)
        if phi == 1:
            return None, False

    for _ in range(n_sub):
        k1 = dinamica_reator_r101(x, q, u_cont)
        k2 = dinamica_reator_r101(x + 0.5 * h * k1, q, u_cont)
        k3 = dinamica_reator_r101(x + 0.5 * h * k2, q, u_cont)
        k4 = dinamica_reator_r101(x + h * k3, q, u_cont)
        x = x + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

        # Saturação física inferior
        x[0] = max(0.0, x[0])
        x[1] = max(20.0, x[1])
        x[2] = np.clip(x[2], 0.0, 100.0)
        x[3] = max(0.0, x[3])

        if not ignorar_sis:
            phi, _ = avaliar_sis_booleano(x)
            if phi == 1:
                return None, False
        traj.append(x.copy())

    return np.array(traj), True

print("Modelo dinâmico não linear e verificador booleano SIS prontos.")

Modelo dinâmico não linear e verificador booleano SIS prontos.


### 3. Métrica Híbrida e Algoritmo *Kinodynamic RRT*

Para medir proximidade entre dois estados híbridos $\mathbf{z}_a = (\mathbf{x}_a, \mathbf{q}_a)$ e $\mathbf{z}_b = (\mathbf{x}_b, \mathbf{q}_b)$ na etapa **`NEAREST`**, combinamos a norma euclidiana ponderada das variáveis contínuas com a **distância de Hamming** $d_H(\mathbf{q}_a, \mathbf{q}_b)$ dos atuadores booleanos:

$$d(\mathbf{z}_a, \mathbf{z}_b) = \left\| \mathbf{W}(\mathbf{x}_a - \mathbf{x}_b) \right\|_2 + \lambda \sum_{i=1}^{3} \mathbb{I}(q_{a,i} \neq q_{b,i})$$

O parâmetro $\lambda > 0$ penaliza chaveamentos desnecessários de válvulas solenoide e motores (*chattering* mecânico). O grafo resultante $\mathcal{T} = (V, E)$ é construído como uma **arborescência direcionada** enraizada em $\mathbf{z}_{\text{inicial}}$.

In [3]:
SCALES = np.array([180.0, 200.0, 70.0, 25.0, 1.0])

def distancia_hibrida(z1, z2, lambda_hamming=0.15):
    """Métrica híbrida: norma euclidiana normalizada + distância de Hamming."""
    x1, q1 = z1
    x2, q2 = z2
    d_cont = np.linalg.norm((x1 - x2) / SCALES)
    d_hamming = sum(a != b for a, b in zip(q1, q2))
    return d_cont + lambda_hamming * d_hamming


def executar_rrt_reator(
    z_init,
    x_goal,
    tol_goal,
    k_max=3000,
    p_goal=0.10,
    n_candidates=10,
    dt=1.0,
    seed=2026
):
    """
    Sintetiza uma trajetória segura de transição no Reator R-101 usando
    Kinodynamic RRT sobre o grafo direcionado T = (V, E).
    """
    rng = np.random.default_rng(seed)
    t0 = time.perf_counter()

    # Inicialização do grafo direcionado T = (V, E)
    T = nx.DiGraph()
    T.add_node(0, x=z_init[0], q=z_init[1], time=0.0)

    # Caixa de amostragem dentro de Z_livre
    low = np.array([45.0, 65.0, 24.0, 0.0, 0.85])
    high = np.array([176.0, 196.0, 86.0, 23.0, 2.05])

    goal_node = None
    colisoes_sis = 0

    for k in range(1, k_max + 1):
        # 1. SAMPLE (Amostragem aleatória com Goal Bias)
        if rng.random() < p_goal:
            x_rand = x_goal + rng.normal(0.0, tol_goal * 0.35)
            q_rand = (1, 1, 1)
        else:
            x_rand = rng.uniform(low, high)
            q_rand = (
                int(rng.random() < 0.85),
                int(rng.random() < 0.85),
                int(rng.random() < 0.90),
            )
        z_rand = (x_rand, q_rand)

        # 2. NEAREST (Busca do vértice mais próximo em V)
        best_node = None
        best_dist = float("inf")
        for node_id, data in T.nodes(data=True):
            d = distancia_hibrida((data["x"], data["q"]), z_rand)
            if d < best_dist:
                best_dist = d
                best_node = node_id

        x_near = T.nodes[best_node]["x"]
        q_near = T.nodes[best_node]["q"]

        # 3. STEER (Expansão cinodinâmica com candidatos de controle)
        best_traj = None
        best_u = None
        best_q_next = None
        best_steer_dist = float("inf")

        for _ in range(n_candidates):
            u_cand = rng.uniform(0.0, 1.0, size=3)
            q_cand = q_rand if rng.random() < 0.55 else q_near
            traj, safe = integrar_passo_seguro(x_near, q_cand, u_cand, dt=dt)

            if not safe:
                colisoes_sis += 1
                continue

            d_new = distancia_hibrida((traj[-1], q_cand), z_rand)
            if d_new < best_steer_dist:
                best_steer_dist = d_new
                best_traj = traj
                best_u = u_cand
                best_q_next = q_cand

        # 4. COLLISION_FREE & EXPANSÃO DO GRAFO T = (V, E)
        if best_traj is not None:
            new_id = T.number_of_nodes()
            x_new = best_traj[-1]
            T.add_node(
                new_id,
                x=x_new,
                q=best_q_next,
                time=T.nodes[best_node]["time"] + dt
            )
            T.add_edge(
                best_node,
                new_id,
                u=best_u,
                q=best_q_next,
                traj=best_traj,
                dt=dt
            )

            # 5. VERIFICAÇÃO DE CONVERGÊNCIA PARA A REGIÃO ALVO
            erro_norm = np.abs(x_new - x_goal) / tol_goal
            if np.all(erro_norm <= 1.0) and best_q_next == (1, 1, 1):
                goal_node = new_id
                break

    elapsed = time.perf_counter() - t0
    return T, goal_node, colisoes_sis, elapsed

print("Função executar_rrt_reator definida.")

Função executar_rrt_reator definida.


### 4. Execução da Simulação Sintética e Propriedades do Grafo Discreto

Definimos o seguinte cenário operacional no reator `R-101`:
- **Estado Inicial ($\mathbf{z}_{\text{inicial}}$ — Regime MAP):** $P = 72{,}0\text{ bar}$, $T = 108{,}0^\circ\text{C}$, $L = 42{,}0\%$, $C_{\text{NH}_3} = 3{,}0\text{ ppm}$, $\alpha = 1{,}02$, com atuadores $\mathbf{q}_0 = (1, 1, 1)$.
- **Região Alvo ($\mathcal{Z}_{\text{alvo}}$ — Regime DAP):** $P \approx 145\text{ bar}$, $T \approx 178^\circ\text{C}$, $L \approx 68\%$, $C_{\text{NH}_3} \approx 11\text{ ppm}$, $\alpha \approx 1{,}85$.

In [4]:
# Condição inicial (Campanha MAP) e objetivo (Campanha DAP)
z_inicial = (np.array([72.0, 108.0, 42.0, 3.0, 1.02]), (1, 1, 1))
x_objetivo = np.array([145.0, 178.0, 68.0, 11.0, 1.85])
tolerancia = np.array([10.0, 8.0, 8.0, 6.0, 0.05])

# Execução do planejador RRT
T, goal_node, colisoes_sis, tempo_exec = executar_rrt_reator(
    z_init=z_inicial,
    x_goal=x_objetivo,
    tol_goal=tolerancia,
    k_max=3000,
    p_goal=0.10,
    n_candidates=10,
    dt=1.0,
    seed=2026
)

# Análise de Matemática Discreta sobre a árvore T = (V, E)
num_v = T.number_of_nodes()
num_e = T.number_of_edges()
is_arborescence = nx.is_arborescence(T)
graus_saida = [d for _, d in T.out_degree()]
profundidade_max = max(nx.shortest_path_length(T, source=0).values())

print("=" * 72)
print("RELATÓRIO DE SÍNTESE RRT & PROPRIEDADES DO GRAFO DISCRETO T = (V, E)")
print("=" * 72)
print(f"Status de Convergência     : {'SUCESSO (Trajetória Segura Encontrada)' if goal_node else 'FALHA'}")
print(f"Tempo de Computação        : {tempo_exec:.3f} s")
print(f"Cardinalidade |V| e |E|    : |V| = {num_v} vértices, |E| = {num_e} arestas (|E| = |V| - 1: {num_e == num_v - 1})")
print(f"É Arborescência Enraizada? : {is_arborescence}")
print(f"Profundidade Máxima da Árv.: {profundidade_max} passos")
print(f"Grau de Saída Máximo       : {max(graus_saida)} (Média: {np.mean(graus_saida):.2f})")
print(f"Ramos Podados pelo SIS     : {colisoes_sis} tentativas bloqueadas (Phi_SIS = 1)")

# Reconstrução da trajetória (BACKTRACK)
caminho_vertices = nx.shortest_path(T, source=0, target=goal_node)
print(f"Comprimento da Rota Ótima  : {len(caminho_vertices) - 1} passos ({(len(caminho_vertices) - 1)*1.0:.1f} min)")
print("-" * 72)
print(f"{'Passo':>5} | {'t(min)':>6} | {'P(bar)':>7} | {'T(°C)':>7} | {'L(%)':>6} | {'NH3(ppm)':>8} | {'alpha':>6} | {'u_NH3':>5} | {'u_H3PO4':>7} | {'u_cool':>6} | {'q=(v1,v2,m1)'}")
print("-" * 72)

for idx, v in enumerate(caminho_vertices):
    x_v = T.nodes[v]["x"]
    q_v = T.nodes[v]["q"]
    t_v = T.nodes[v]["time"]
    if idx < len(caminho_vertices) - 1:
        edge_data = T.edges[v, caminho_vertices[idx + 1]]
        u_str = f"{edge_data['u'][0]:5.2f} | {edge_data['u'][1]:7.2f} | {edge_data['u'][2]:6.2f}"
    else:
        u_str = "  --- |     --- |    ---"
    print(f"{idx:5d} | {t_v:6.1f} | {x_v[0]:7.2f} | {x_v[1]:7.2f} | {x_v[2]:6.2f} | {x_v[3]:8.2f} | {x_v[4]:6.3f} | {u_str} | {q_v}")

RELATÓRIO DE SÍNTESE RRT & PROPRIEDADES DO GRAFO DISCRETO T = (V, E)
Status de Convergência     : SUCESSO (Trajetória Segura Encontrada)
Tempo de Computação        : 1.234 s
Cardinalidade |V| e |E|    : |V| = 77 vértices, |E| = 76 arestas (|E| = |V| - 1: True)
É Arborescência Enraizada? : True
Profundidade Máxima da Árv.: 14 passos
Grau de Saída Máximo       : 4 (Média: 0.99)
Ramos Podados pelo SIS     : 38 tentativas bloqueadas (Phi_SIS = 1)
Comprimento da Rota Ótima  : 13 passos (13.0 min)
------------------------------------------------------------------------
Passo | t(min) |  P(bar) |   T(°C) |   L(%) | NH3(ppm) |  alpha | u_NH3 | u_H3PO4 | u_cool | q=(v1,v2,m1)
------------------------------------------------------------------------
    0 |    0.0 |   72.00 |  108.00 |  42.00 |     3.00 |  1.020 |  0.61 |    0.35 |   0.95 | (1, 1, 1)
    1 |    1.0 |   68.43 |  107.17 |  41.16 |     2.85 |  1.140 |  0.92 |    0.19 |   0.94 | (1, 1, 1)
    2 |    2.0 |   68.29 |   99.14 |  41.22 |

### 5. Visualização da Exploração RRT nos Planos de Fase do Reator `R-101`

Abaixo projetamos a árvore discreta $\mathcal{T} = (V, E)$ em dois planos de fase críticos da engenharia do processo:
1. **Plano Termodinâmico ($T \times P$):** Evidencia como os ramos do RRT exploram o espaço livre e contornam as barreiras críticas de sobretemperatura (`TT-101` $\ge 200^\circ\text{C}$) e sobrepressão (`PT-101` $\ge 180\text{ bar}$).
2. **Plano Operacional ($L \times \alpha$):** Mostra a evolução da razão estequiométrica $\text{N/P}$ de MAP ($1{,}02$) para DAP ($1{,}85$) respeitando os limites de cavitação (`LT-101B` $\le 20\%$) e transbordamento (`LT-101A` $\ge 90\%$).

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- SUBPLOT 1: Plano de Fase Termodinâmico (Temperatura T x Pressão P) ---
ax1 = axes[0]

# Regiões proibidas pelo SIS (Z_obs)
ax1.axvspan(SIS_LIMITS["T_max"], 230, color="#d62728", alpha=0.18, label="Zona Proibida SIS (TT-101 / PT-101)")
ax1.axhspan(SIS_LIMITS["P_max"], 210, color="#d62728", alpha=0.18)
ax1.axvline(SIS_LIMITS["T_max"], color="#d62728", linestyle="--", linewidth=1.6)
ax1.axhline(SIS_LIMITS["P_max"], color="#d62728", linestyle="--", linewidth=1.6)

# Todas as arestas exploradas pela árvore RRT
for u_node, v_node, data in T.edges(data=True):
    traj = data["traj"]
    ax1.plot(traj[:, 1], traj[:, 0], color="#4a90e2", alpha=0.35, linewidth=1.1)

# Vértices da árvore
xs_all = np.array([data["x"] for _, data in T.nodes(data=True)])
ax1.scatter(xs_all[:, 1], xs_all[:, 0], color="#2b5c8f", s=16, alpha=0.65, zorder=3, label="Vértices RRT (V)")

# Região Alvo (Campanha DAP)
rect_dap = patches.Rectangle(
    (x_objetivo[1] - tolerancia[1], x_objetivo[0] - tolerancia[0]),
    2 * tolerancia[1],
    2 * tolerancia[0],
    linewidth=1.5,
    edgecolor="#2ca02c",
    facecolor="#2ca02c",
    alpha=0.20,
    label="Região Alvo DAP (Z_alvo)"
)
ax1.add_patch(rect_dap)

# Trajetória extraída (Backtrack)
for idx in range(len(caminho_vertices) - 1):
    traj_opt = T.edges[caminho_vertices[idx], caminho_vertices[idx + 1]]["traj"]
    lbl = "Trajetória Segura RRT" if idx == 0 else None
    ax1.plot(traj_opt[:, 1], traj_opt[:, 0], color="#ff7f0e", linewidth=3.0, zorder=5, label=lbl)

ax1.scatter(z_inicial[0][1], z_inicial[0][0], color="#1f77b4", edgecolor="black", s=110, zorder=6, label="Início: Regime MAP")
x_final_rrt = T.nodes[goal_node]["x"]
ax1.scatter(x_final_rrt[1], x_final_rrt[0], color="#2ca02c", edgecolor="black", s=120, marker="*", zorder=7, label="Fim: Regime DAP Alcançado")

ax1.set_title("Plano de Fase Termodinâmico: Temperatura vs. Pressão", fontweight="bold")
ax1.set_xlabel("Temperatura TT-101 (°C)")
ax1.set_ylabel("Pressão PT-101 (bar)")
ax1.set_xlim(95, 220)
ax1.set_ylim(60, 200)
ax1.legend(loc="upper left", fontsize=8.5, framealpha=0.92)

# --- SUBPLOT 2: Plano Operacional (Nível L x Razão Molar alpha) ---
ax2 = axes[1]

ax2.axhspan(0, SIS_LIMITS["L_min"], color="#d62728", alpha=0.18, label="Zona Proibida SIS (LT-101A/B)")
ax2.axhspan(SIS_LIMITS["L_max"], 100, color="#d62728", alpha=0.18)
ax2.axhline(SIS_LIMITS["L_min"], color="#d62728", linestyle="--", linewidth=1.6)
ax2.axhline(SIS_LIMITS["L_max"], color="#d62728", linestyle="--", linewidth=1.6)

for u_node, v_node, data in T.edges(data=True):
    traj = data["traj"]
    ax2.plot(traj[:, 4], traj[:, 2], color="#4a90e2", alpha=0.35, linewidth=1.1)

ax2.scatter(xs_all[:, 4], xs_all[:, 2], color="#2b5c8f", s=16, alpha=0.65, zorder=3)

rect_dap2 = patches.Rectangle(
    (x_objetivo[4] - tolerancia[4], x_objetivo[2] - tolerancia[2]),
    2 * tolerancia[4],
    2 * tolerancia[2],
    linewidth=1.5,
    edgecolor="#2ca02c",
    facecolor="#2ca02c",
    alpha=0.20,
    label="Região Alvo DAP (alpha ~ 1.85)"
)
ax2.add_patch(rect_dap2)

for idx in range(len(caminho_vertices) - 1):
    traj_opt = T.edges[caminho_vertices[idx], caminho_vertices[idx + 1]]["traj"]
    lbl = "Trajetória Segura RRT" if idx == 0 else None
    ax2.plot(traj_opt[:, 4], traj_opt[:, 2], color="#ff7f0e", linewidth=3.0, zorder=5, label=lbl)

ax2.scatter(z_inicial[0][4], z_inicial[0][2], color="#1f77b4", edgecolor="black", s=110, zorder=6, label="Início: MAP (alpha = 1.02)")
ax2.scatter(x_final_rrt[4], x_final_rrt[2], color="#2ca02c", edgecolor="black", s=120, marker="*", zorder=7, label="Fim: DAP (alpha = 1.85)")

ax2.set_title("Plano Operacional: Estequiometria N/P vs. Nível do Slurry", fontweight="bold")
ax2.set_xlabel("Razão Molar Estequiométrica N/P (alpha)")
ax2.set_ylabel("Nível Volumétrico LT-101 (%)")
ax2.set_xlim(0.90, 2.05)
ax2.set_ylim(10, 100)
ax2.legend(loc="lower right", fontsize=8.5, framealpha=0.92)

plt.tight_layout()
plt.show()

### 6. Validação de Segurança: Trajetória RRT vs. Manobra Direta em Malha Aberta

Para evidenciar a importância do planejamento cinodinâmico com restrições booleanas, comparamos a sequência de controle sintetizada pelo **RRT** contra uma **manobra convencional em malha aberta**, na qual o operador aplica diretamente as vazões nominais de regime DAP ($u_{\text{NH}_3} = 0{,}88$, $u_{\text{H}_3\text{PO}_4} = 0{,}45$, $u_{\text{cool}} = 0{,}30$).

Como observado abaixo, a manobra direta provoca um surto exotérmico que ultrapassa $200^\circ\text{C}$ em menos de $5\text{ min}$, disparando o alarme `ALM-101` e o fechamento *fail-safe* do SIS ($\Phi_{\text{SIS}} = 1$), enquanto a trajetória RRT mantém todos os sensores estritamente na faixa segura ($\Phi_{\text{SIS}} = 0$).

In [6]:
# 1. Reconstruir série temporal contínua da trajetória RRT
t_rrt = []
x_rrt = []
u_rrt = []

t_acum = 0.0
for idx in range(len(caminho_vertices) - 1):
    edge = T.edges[caminho_vertices[idx], caminho_vertices[idx + 1]]
    traj_seg = edge["traj"]
    dt_seg = edge["dt"]
    ts = np.linspace(t_acum, t_acum + dt_seg, len(traj_seg))
    if idx > 0:
        ts = ts[1:]
        traj_seg = traj_seg[1:]
    t_rrt.extend(ts)
    x_rrt.extend(traj_seg)
    u_rrt.append((t_acum, edge["u"]))
    t_acum += dt_seg

t_rrt = np.array(t_rrt)
x_rrt = np.array(x_rrt)

# 2. Simular manobra convencional em malha aberta (degrau direto para DAP)
u_degrau = np.array([0.88, 0.45, 0.30])
traj_aberta, _ = integrar_passo_seguro(
    z_inicial[0],
    (1, 1, 1),
    u_degrau,
    dt=t_acum,
    n_sub=len(t_rrt) - 1,
    ignorar_sis=True
)

# Identificar instante de disparo do SIS na malha aberta
trip_idx = next((i for i, x_val in enumerate(traj_aberta) if avaliar_sis_booleano(x_val)[0] == 1), None)

# 3. Plotagem comparativa dos 4 transmissores ISA-5.1
fig, axes = plt.subplots(2, 2, figsize=(15, 8.5), sharex=True)

configs = [
    (0, "PT-101: Pressão do Reator (bar)", SIS_LIMITS["P_max"], None),
    (1, "TT-101: Temperatura do Slurry (°C)", SIS_LIMITS["T_max"], None),
    (2, "LT-101A/B: Nível Volumétrico (%)", SIS_LIMITS["L_max"], SIS_LIMITS["L_min"]),
    (3, "AT-101: Amônia Ambiente (ppm)", SIS_LIMITS["C_nh3_max"], None),
]

for ax, (col_idx, titulo, lim_sup, lim_inf) in zip(axes.flat, configs):
    ax.plot(t_rrt, x_rrt[:, col_idx], color="#ff7f0e", linewidth=2.6, label="Trajetória RRT (Segura)")
    ax.plot(t_rrt, traj_aberta[:, col_idx], color="#7f7f7f", linestyle="-.", linewidth=2.0, label="Degrau Malha Aberta")

    if lim_sup is not None:
        ax.axhline(lim_sup, color="#d62728", linestyle="--", linewidth=1.5, label=f"Limite SIS ({lim_sup})")
    if lim_inf is not None:
        ax.axhline(lim_inf, color="#d62728", linestyle=":", linewidth=1.5, label=f"Limite Mín. SIS ({lim_inf})")

    if trip_idx is not None:
        ax.scatter(
            t_rrt[trip_idx],
            traj_aberta[trip_idx, col_idx],
            color="#d62728",
            s=90,
            zorder=6,
            label="Trip SIS (Phi_SIS = 1)" if col_idx == 1 else None
        )

    ax.set_title(titulo, fontweight="bold")
    ax.legend(loc="best", fontsize=8.5)

axes[1, 0].set_xlabel("Tempo de Operação (min)")
axes[1, 1].set_xlabel("Tempo de Operação (min)")

plt.suptitle("Monitoramento SCADA-Core: Trajetória Sintetizada pelo RRT vs. Degrau Convencional", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

if trip_idx is not None:
    _, props_trip = avaliar_sis_booleano(traj_aberta[trip_idx])
    ativos = [k for k, v in props_trip.items() if v == 1]
    print(f"[ALERTA SCADA-Core] Na malha aberta, o SIS disparou em t = {t_rrt[trip_idx]:.2f} min! Proposições ativas: {ativos}")
print(f"[OK SCADA-Core] Na trajetória RRT, Phi_SIS permaneceu igual a 0 durante todos os {t_rrt[-1]:.1f} min de transição.")

[ALERTA SCADA-Core] Na malha aberta, o SIS disparou em t = 3.30 min! Proposições ativas: ['t1']
[OK SCADA-Core] Na trajetória RRT, Phi_SIS permaneceu igual a 0 durante todos os 13.0 min de transição.


### 7. Síntese dos Resultados

1. **Integração Controle–Matemática Discreta:** O algoritmo RRT converteu as restrições termodinâmicas e de segurança funcional da norma **ISA-5.1** em barreiras booleanas ($\Phi_{\text{SIS}}$), construindo uma árvore direcionada $\mathcal{T}=(V,E)$ livre de colisões em menos de $1{,}5\text{ s}$.
2. **Viabilidade Cinodinâmica:** Cada aresta do grafo corresponde à integração numérica real das equações não lineares do reator `R-101`, garantindo que os sinais de controle ($u_{\text{NH}_3}, u_{\text{H}_3\text{PO}_4}, u_{\text{cool}}$) respeitem a inércia térmica e capacitiva da planta.
3. **Extensões Possíveis:**
   - Substituir o RRT padrão pelo **RRT\*** (*rewiring*) para minimizar o consumo energético de resfriamento ($\int u_{\text{cool}}\,dt$) ou o tempo total de transição;
   - Utilizar a trajetória extraída como referência (*setpoint* dinâmico) para um controlador preditivo (**NMPC**) em malha fechada contra perturbações na concentração do ácido fosfórico de entrada.